#БЛОК 1

In [1]:
import json
import glob
import numpy as np
import pandas as pd
from tqdm import tqdm
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, precision_recall_curve
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.ensemble import VotingClassifier
from sklearn.calibration import CalibratedClassifierCV
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

print("✅ Библиотеки импортированы")

✅ Библиотеки импортированы


#БЛОК 2

In [2]:
def load_data(base_path):
    """
    Загрузка данных с ПРАВИЛЬНОЙ структурой файлов.
    Каждая строка = JSON массив, где [2] содержит транзакцию.
    """
    rows, labels, file_ids = [], [], []
    
    for label, cls in enumerate(['clean', 'malware']):
        cls_path = Path(base_path) / cls
        files = list(cls_path.glob('*.jsonl'))
        print(f"📁 {cls}: {len(files)} файлов")
        
        for fpath in tqdm(files, desc=f"Processing {cls}"):
            file_transactions = []
            
            try:
                with open(fpath, 'r', encoding='utf-8') as f:
                    for line in f:
                        line = line.strip()
                        if not line:
                            continue
                        try:
                            obj = json.loads(line)
                            
                            # ВАЖНО: obj — это массив, транзакция в [2]
                            if isinstance(obj, list) and len(obj) > 2:
                                tx = obj[2]
                                rqs = tx.get('rqs', {})
                                rsp = tx.get('rsp', {})
                                
                                # Создаём текстовое представление транзакции
                                text_parts = []
                                for k, v in rqs.items():
                                    if v is not None:
                                        text_parts.append(f"{k}:{v}")
                                for k, v in rsp.items():
                                    if v is not None:
                                        text_parts.append(f"{k}:{v}")
                                
                                text = " ".join(text_parts)
                                
                                # Фильтр: пропускаем слишком короткие транзакции
                                if len(text) > 20:
                                    file_transactions.append((text, label, fpath.stem))
                                    
                        except (json.JSONDecodeError, IndexError, KeyError):
                            continue
                            
                # Добавляем все транзакции из файла
                for text, label, file_id in file_transactions:
                    rows.append(text)
                    labels.append(label)
                    file_ids.append(file_id)
                    
            except Exception as e:
                continue
    
    print(f"✅ Всего транзакций: {len(rows)}")
    return pd.DataFrame({'text': rows, 'label': labels, 'file_id': file_ids})

#БЛОК 3


In [3]:
print("=== 📦 Загрузка обучающих данных ===")
df = load_data('/kaggle/input/competitions/http-malware-detection/train')

print(f"\n📊 Распределение классов:")
print(df['label'].value_counts())
print(f"Дисбаланс: {(df['label']==0).sum()/(df['label']==1).sum():.1f}:1")

# Разделение на train/validation
train_texts, val_texts, y_train, y_val = train_test_split(
    df['text'], 
    df['label'], 
    test_size=0.2, 
    random_state=42, 
    stratify=df['label']
)

print(f"\nTrain: {len(train_texts)} транзакций")
print(f"Val: {len(val_texts)} транзакций")

=== 📦 Загрузка обучающих данных ===
📁 clean: 3419 файлов


Processing clean: 100%|██████████| 3419/3419 [00:15<00:00, 224.18it/s]


📁 malware: 1016 файлов


Processing malware: 100%|██████████| 1016/1016 [00:04<00:00, 237.76it/s]

✅ Всего транзакций: 13111

📊 Распределение классов:
label
0    8178
1    4933
Name: count, dtype: int64
Дисбаланс: 1.7:1

Train: 10488 транзакций
Val: 2623 транзакций


#БЛОК 4


In [4]:
print("\n=== 🔤 TF-IDF векторизация ===")

vectorizer = TfidfVectorizer(
    max_features=5000,        # Увеличено с 1000
    ngram_range=(1, 4),       # Оптимизировано
    min_df=3,                 # Минимум 3 документа
    max_df=0.95,              # Максимум 95% документов
    sublinear_tf=True,        # Логарифмическое масштабирование
    strip_accents='unicode',  # Удаление акцентов
    lowercase=True            # Приведение к нижнему регистру
)

X_train = vectorizer.fit_transform(train_texts)
X_val = vectorizer.transform(val_texts)

print(f"📐 Форма train: {X_train.shape}")
print(f"📐 Форма val: {X_val.shape}")


=== 🔤 TF-IDF векторизация ===
📐 Форма train: (10488, 5000)
📐 Форма val: (2623, 5000)


#БЛОК 5


In [5]:
print("\n=== 📊 Кросс-валидация (5-fold) ===")

def cv_with_retraining(X, y, model_class, params, n_splits=5):
    """Правильная CV с переобучением модели на каждом фолде"""
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    oof_proba = np.zeros(len(X))
    f1_scores = []
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
        X_tr, X_val_fold = X[train_idx], X[val_idx]
        y_tr, y_val_fold = y.iloc[train_idx], y.iloc[val_idx]
        
        # Переобучаем модель на фолде
        model = model_class(**params)
        model.fit(X_tr, y_tr)
        
        # Предсказания на валидации
        oof_proba[val_idx] = model.predict_proba(X_val_fold)[:, 1]
        fold_pred = (oof_proba[val_idx] > 0.5).astype(int)
        fold_f1 = f1_score(y_val_fold, fold_pred)
        f1_scores.append(fold_f1)
        
        print(f"Fold {fold}: F1 = {fold_f1:.4f}")
    
    mean_f1 = np.mean(f1_scores)
    print(f"\n📊 Mean CV F1: {mean_f1:.4f} (+/- {np.std(f1_scores):.4f})")
    return oof_proba, mean_f1

# Параметры для LogisticRegression
lr_params = {
    'max_iter': 500,
    'C': 1.0,
    'class_weight': 'balanced',
    'solver': 'lbfgs',
    'random_state': 42
}

# Параметры для LightGBM
lgb_params = {
    'n_estimators': 500,
    'learning_rate': 0.03,
    'max_depth': 10,
    'num_leaves': 50,
    'class_weight': 'balanced',
    'random_state': 42,
    'verbose': -1
}

# CV для LogisticRegression
print("\n--- LogisticRegression CV ---")
oof_proba_lr, cv_f1_lr = cv_with_retraining(
    X_train.toarray(), y_train, LogisticRegression, lr_params
)

# CV для LightGBM
print("\n--- LightGBM CV ---")
oof_proba_lgb, cv_f1_lgb = cv_with_retraining(
    X_train.toarray(), y_train, lgb.LGBMClassifier, lgb_params
)


=== 📊 Кросс-валидация (5-fold) ===

--- LogisticRegression CV ---
Fold 1: F1 = 0.8712
Fold 2: F1 = 0.8530
Fold 3: F1 = 0.8623
Fold 4: F1 = 0.8787
Fold 5: F1 = 0.8744

📊 Mean CV F1: 0.8679 (+/- 0.0092)

--- LightGBM CV ---
Fold 1: F1 = 0.9030
Fold 2: F1 = 0.8877
Fold 3: F1 = 0.8939
Fold 4: F1 = 0.9156
Fold 5: F1 = 0.9104

📊 Mean CV F1: 0.9021 (+/- 0.0103)


#БЛОК 6


In [6]:
print("\n=== ⚡ Оптимизация порога ===")

def find_optimal_threshold(y_true, y_proba):
    """Находим порог на OOF predictions (не на public LB!)"""
    precisions, recalls, thresholds = precision_recall_curve(y_true, y_proba)
    f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-10)
    
    optimal_idx = f1_scores.argmax()
    optimal_threshold = thresholds[optimal_idx] if optimal_idx < len(thresholds) else 0.5
    best_f1 = f1_scores[optimal_idx]
    
    return optimal_threshold, best_f1

# Оптимизация для LR
threshold_lr, f1_lr = find_optimal_threshold(y_train, oof_proba_lr)
print(f"LR: Порог = {threshold_lr:.4f}, CV F1 = {f1_lr:.4f}")

# Оптимизация для LGB
threshold_lgb, f1_lgb = find_optimal_threshold(y_train, oof_proba_lgb)
print(f"LGB: Порог = {threshold_lgb:.4f}, CV F1 = {f1_lgb:.4f}")

# Ensemble OOF (среднее вероятностей)
oof_proba_ensemble = (oof_proba_lr + oof_proba_lgb) / 2
threshold_ensemble, f1_ensemble = find_optimal_threshold(y_train, oof_proba_ensemble)
print(f"Ensemble: Порог = {threshold_ensemble:.4f}, CV F1 = {f1_ensemble:.4f}")

FINAL_THRESHOLD = threshold_ensemble
print(f"\n🎯 Используем порог: {FINAL_THRESHOLD:.4f}")


=== ⚡ Оптимизация порога ===
LR: Порог = 0.4936, CV F1 = 0.8683
LGB: Порог = 0.5391, CV F1 = 0.9039
Ensemble: Порог = 0.5204, CV F1 = 0.8995

🎯 Используем порог: 0.5204


#БЛОК 7


In [7]:
print("\n=== 🎯 Финальное обучение моделей ===")

# Обучение LogisticRegression на всех данных
print("Обучение LogisticRegression...")
clf_lr = LogisticRegression(**lr_params)
clf_lr.fit(X_train, y_train)

# Обучение LightGBM на всех данных
print("Обучение LightGBM...")
clf_lgb = lgb.LGBMClassifier(**lgb_params)
clf_lgb.fit(X_train.toarray(), y_train)

print("✅ Обе модели обучены!")


=== 🎯 Финальное обучение моделей ===
Обучение LogisticRegression...
Обучение LightGBM...
✅ Обе модели обучены!


#БЛОК 8


In [8]:
def predict_file(file_path, vectorizer, clf_lr, clf_lgb, threshold=0.5):
    """
    Предсказание для файла: если ХОТЯ БЫ ОДНА транзакция = malware → файл = malware
    """
    preds = []
    probas = []
    
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    obj = json.loads(line)
                    
                    if isinstance(obj, list) and len(obj) > 2:
                        tx = obj[2]
                        rqs = tx.get('rqs', {})
                        rsp = tx.get('rsp', {})
                        
                        text_parts = []
                        for k, v in rqs.items():
                            if v is not None:
                                text_parts.append(f"{k}:{v}")
                        for k, v in rsp.items():
                            if v is not None:
                                text_parts.append(f"{k}:{v}")
                        
                        text = " ".join(text_parts)
                        
                        if len(text) > 20:
                            X = vectorizer.transform([text])
                            
                            # Ensemble prediction
                            proba_lr = clf_lr.predict_proba(X)[0][1]
                            proba_lgb = clf_lgb.predict_proba(X.toarray())[0][1]
                            proba_ensemble = (proba_lr + proba_lgb) / 2
                            
                            probas.append(proba_ensemble)
                            preds.append(1 if proba_ensemble >= threshold else 0)
                            
                except (json.JSONDecodeError, IndexError, KeyError):
                    continue
        
        # Если хотя бы одна транзакция = malware → файл = malware
        if len(preds) == 0:
            return 0  # Нет транзакций = clean (консервативно)
        
        return int(any(pred == 1 for pred in preds))
        
    except Exception as e:
        return 0

#БЛОК 9


In [9]:
print("\n=== 🔮 Генерация предсказаний ===")
test_path = Path('/kaggle/input/competitions/http-malware-detection/test')
test_files = list(test_path.glob('*.jsonl'))

print(f"📁 Test файлов: {len(test_files)}")

result = []
for path in tqdm(test_files, desc="Predicting"):
    predict = predict_file(path, vectorizer, clf_lr, clf_lgb, threshold=FINAL_THRESHOLD)
    result.append({'id': Path(path).stem, 'target': predict})

result_df = pd.DataFrame(result)
result_df.to_csv('submission.csv', index=False)

print(f"\n✅ submission.csv сохранён!")
print(f"📊 Распределение предсказаний:")
print(result_df['target'].value_counts())
print(f"  Malware: {(result_df['target']==1).sum()/len(result_df)*100:.1f}%")
print(f"  Clean: {(result_df['target']==0).sum()/len(result_df)*100:.1f}%")


=== 🔮 Генерация предсказаний ===
📁 Test файлов: 1326


Predicting: 100%|██████████| 1326/1326 [00:58<00:00, 22.75it/s]


✅ submission.csv сохранён!
📊 Распределение предсказаний:
target
0    947
1    379
Name: count, dtype: int64
  Malware: 28.6%
  Clean: 71.4%


#БЛОК 10


In [10]:
print("\n=== 📊 Валидация на hold-out set ===")

# Предсказания на валидации
y_proba_val_lr = clf_lr.predict_proba(X_val)[:, 1]
y_proba_val_lgb = clf_lgb.predict_proba(X_val.toarray())[:, 1]
y_proba_val_ensemble = (y_proba_val_lr + y_proba_val_lgb) / 2

# F1 с оптимизированным порогом
y_pred_val = (y_proba_val_ensemble >= FINAL_THRESHOLD).astype(int)
val_f1 = f1_score(y_val, y_pred_val)

# F1 с default порогом
y_pred_default = (y_proba_val_ensemble >= 0.5).astype(int)
default_f1 = f1_score(y_val, y_pred_default)

print(f"Hold-out F1 (optimized threshold): {val_f1:.4f}")
print(f"Hold-out F1 (default threshold=0.5): {default_f1:.4f}")
print(f"Улучшение от оптимизации порога: +{(val_f1 - default_f1)*100:.2f}%")


=== 📊 Валидация на hold-out set ===
Hold-out F1 (optimized threshold): 0.9072
Hold-out F1 (default threshold=0.5): 0.9054
Улучшение от оптимизации порога: +0.18%
